# Supervised Learning: Identifying Crypto Mining Data Centers

The FracTracker data center list does not say which facilities are crypto mines. The separate FracTracker crypto tracker does, but the two files do not share an ID.

This notebook starts by joining them on location (`lat`, `long`): if a data center sits close enough to a known crypto site, we label it as crypto.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from haversine import haversine_vector, Unit

In [ ]:
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load data

The data centers come from `02-data-cleaning.ipynb`.

The crypto file has not been cleaned yet, so it still comes from `data/raw/`.

In [ ]:
data_centers = pd.read_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv")
crypto = pd.read_csv(RAW_DIR / "fractracker_crypto.csv")

print("Data centers:", data_centers.shape)
print("Crypto sites:", crypto.shape)

### Drop invalid coordinates

A few rows have coordinates that cannot be real: a longitude of `-11402.57827` (likely a typo) and a data center at `0, 0` (missing location). We can't measure distance for these, so we drop them.

In [ ]:
def valid_coordinates(df):
    return df["lat"].between(-90, 90) & df["long"].between(-180, 180) & ~((df["lat"] == 0) & (df["long"] == 0))

print("Dropping data centers:", data_centers.loc[~valid_coordinates(data_centers), "facility_name"].tolist())
print("Dropping crypto sites:", crypto.loc[~valid_coordinates(crypto), "Name"].tolist())

data_centers = data_centers[valid_coordinates(data_centers)].reset_index(drop=True)
crypto = crypto[valid_coordinates(crypto)].reset_index(drop=True)

## Distance between every crypto site and every data center

The `haversine` package takes `(lat, long)` points in degrees and returns the distance along the Earth's surface. `haversine_vector` with `comb=True` compares every point in the first list with every point in the second.

The result is a table with one row per crypto site and one column per data center.

In [ ]:
# comb=True returns one row per data center, so .T flips it to one row per crypto site
distances = haversine_vector(
    crypto[["lat", "long"]].values,
    data_centers[["lat", "long"]].values,
    Unit.MILES,
    comb=True,
).T

print("Distance table (crypto sites, data centers):", distances.shape)

## Nearest data center for each crypto site

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [ ]:
nearest_dc = distances.argmin(axis=1)

crypto["nearest_dc_miles"] = distances.min(axis=1)
crypto["nearest_dc_name"] = data_centers["facility_name"].values[nearest_dc]
crypto["nearest_dc_operator"] = data_centers["operator_name"].values[nearest_dc]

for miles in [0.1, 0.25, 0.5, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = crypto[crypto["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Most crypto sites have no data center nearby. Many crypto mines are simply not in the data center list.

The table below puts the names side by side so we can check whether close pairs are really the same facility.

In [ ]:
columns = [
    "Name", "Company", "Precise Location?",
    "nearest_dc_name", "nearest_dc_operator", "nearest_dc_miles"
]

crypto.loc[crypto["nearest_dc_miles"] <= 2, columns].sort_values("nearest_dc_miles")

## Label the data centers

We label from the data center side: for each data center, find the closest crypto site. If it is within `MAX_MILES`, the data center gets `is_crypto = 1`.

Labeling this way handles two things:
- The crypto file repeats some sites (same name or same coordinates), and several crypto rows can point to one data center.
- Every data center gets exactly one label.

`0.5` miles is a starting point. Some sites under 2 miles are clearly different facilities, for example CleanSpark (Clinton) and Amazon Clinton Data Center.

In [ ]:
MAX_MILES = 0.5

nearest_crypto = distances.argmin(axis=0)

data_centers["crypto_miles"] = distances.min(axis=0)
data_centers["crypto_name"] = crypto["Name"].values[nearest_crypto]
data_centers["crypto_company"] = crypto["Company"].values[nearest_crypto]
data_centers["crypto_status"] = crypto["Status"].values[nearest_crypto]

data_centers["is_crypto"] = (data_centers["crypto_miles"] <= MAX_MILES).astype(int)

# only keep crypto details for data centers that matched
not_matched = data_centers["is_crypto"] == 0
data_centers.loc[not_matched, ["crypto_name", "crypto_company", "crypto_status"]] = np.nan

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

In [ ]:
columns = [
    "facility_name", "operator_name", "state", "status",
    "crypto_name", "crypto_company", "crypto_miles"
]

data_centers.loc[data_centers["is_crypto"] == 1, columns].sort_values("crypto_miles")

## How much does the cutoff matter?

The number of positive labels depends on `MAX_MILES`. This helps us choose it.

In [ ]:
rows = []
for miles in [0.1, 0.25, 0.5, 1, 2]:
    labeled_dcs = data_centers[data_centers["crypto_miles"] <= miles]
    matched_crypto = crypto[crypto["nearest_dc_miles"] <= miles]

    rows.append({
        "max_miles": miles,
        "data_centers_labeled": len(labeled_dcs),
        "crypto_sites_matched": len(matched_crypto),
    })

pd.DataFrame(rows)